# Attention Visualization Tutorial

本教程演示 `finesightbench.visualization` 的两大功能：

1. **直接可视化 (Direct Visualization)** — 提取原始注意力权重，生成 heatmap 叠加到原图
2. **Attention Rollout** — 跨层聚合注意力，估算输入 patch 对输出 token 的总体贡献

---

## 目录
- [0. 准备工作](#0-准备工作)
- [1. 模拟注意力数据](#1-模拟注意力数据)
- [2. 直接可视化](#2-直接可视化)
  - [2.1 单层单头注意力图](#21-单层单头注意力图)
  - [2.2 多头平均图](#22-多头平均图)
  - [2.3 多层聚合图](#23-多层聚合图)
  - [2.4 文本 token 条件注意力图 (Cross-Attention)](#24-文本-token-条件注意力图)
- [3. Attention Rollout](#3-attention-rollout)
  - [3.1 视觉编码器 Self-Attention Rollout](#31-视觉编码器-self-attention-rollout)
  - [3.2 Cross-Attention Rollout](#32-cross-attention-rollout)
- [4. 保存与导出](#4-保存与导出)
- [5. 在真实模型上使用 (示例代码框架)](#5-真实模型集成)

## 0. 准备工作

导入需要的库，并关闭 matplotlib 的交互式弹窗（notebook 内嵌显示）。

In [ ]:
import sys, os
# 确保能找到项目根目录
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

# ── 导入可视化模块 ──────────────────────────────────────────────
from finesightbench.visualization import (
    # 直接可视化
    single_head_attention_map,
    multi_head_average_map,
    multi_layer_aggregated_map,
    text_conditioned_attention_map,
    # rollout
    attention_rollout,
    cross_attention_rollout,
    # 叠加 & 显示
    overlay_heatmap,
    save_visualization,
    side_by_side,
)

print("All imports OK ✓")

## 1. 模拟注意力数据

在没有真实模型的情况下，我们先造一组 **合成注意力数据** 来演示所有 API。

我们模拟一个类 ViT-B/16 的配置：
- 输入分辨率 224×224，patch size 16 → patch grid 14×14 = 196 patches
- 加上 CLS token → seq_len = 197
- 12 层，12 heads

为了让可视化更有意义，我们在注意力中注入一个 **热点** —— 模拟模型"看到"了图像中某个区域。

In [ ]:
# ── 基本参数 ────────────────────────────────────────────────────
GRID_H, GRID_W = 14, 14
NUM_PATCHES = GRID_H * GRID_W        # 196
SEQ_LEN = 1 + NUM_PATCHES            # 197 (CLS + patches)
NUM_HEADS = 12
NUM_LAYERS = 12

rng = np.random.default_rng(42)

def make_softmax_attn(shape, rng, hotspot=None, hotspot_weight=5.0):
    """生成合法的注意力权重（每行 softmax 归一化）。
    
    如果指定 hotspot=(row, col)，则在 patch grid 的该位置注入额外权重，
    模拟模型"关注"到图像中的某个区域。
    """
    logits = rng.standard_normal(shape)
    if hotspot is not None:
        r, c = hotspot
        patch_idx = 1 + r * GRID_W + c  # +1 跳过 CLS
        logits[..., patch_idx] += hotspot_weight
    # softmax on last axis
    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    return exp / exp.sum(axis=-1, keepdims=True)

# ── 生成 self-attention 数据 ─────────────────────────────────────
# 我们在右下角 (10, 10) 位置制造一个热点
HOTSPOT = (10, 10)

self_attn = make_softmax_attn(
    (NUM_LAYERS, NUM_HEADS, SEQ_LEN, SEQ_LEN),
    rng, hotspot=HOTSPOT, hotspot_weight=4.0,
)
print(f"self_attn shape: {self_attn.shape}")
print(f"每行和 ≈ 1.0? {np.allclose(self_attn.sum(axis=-1), 1.0)}")

# ── 生成 cross-attention 数据 ────────────────────────────────────
NUM_TEXT_TOKENS = 10  # 假设 10 个文本 token
NUM_CROSS_LAYERS = 3  # 3 层 cross-attention

cross_attn_layers = [
    make_softmax_attn(
        (NUM_HEADS, NUM_TEXT_TOKENS, SEQ_LEN),
        rng, hotspot=HOTSPOT, hotspot_weight=3.0,
    )
    for _ in range(NUM_CROSS_LAYERS)
]
print(f"cross_attn 每层 shape: {cross_attn_layers[0].shape}")
print(f"共 {len(cross_attn_layers)} 层 cross-attention")

### 创建一张测试图像

我们用 FineSightBench 自带的工具画一张简单的带小目标的图片，方便直观检验热力图是否对齐。

In [ ]:
from finesightbench.core.canvas import create_canvas

# 创建 224×224 白色画布
img = create_canvas(224, 224, bg_color=(240, 240, 240))

# 在 hotspot 对应的区域画一个红色小方块
# hotspot (10, 10) 在 14×14 grid 上 → 像素坐标约 (10/14*224, 10/14*224) ≈ (160, 160)
from PIL import ImageDraw
draw = ImageDraw.Draw(img)
cx, cy = int(HOTSPOT[1] / GRID_W * 224) + 8, int(HOTSPOT[0] / GRID_H * 224) + 8
draw.rectangle([cx-4, cy-4, cx+4, cy+4], fill=(255, 0, 0))  # 红色小方块
draw.text((10, 10), "test target", fill=(0, 0, 0))

plt.figure(figsize=(4, 4))
plt.imshow(np.asarray(img))
plt.title("Test Image (red square = hotspot)")
plt.axis("off")
plt.show()

---

## 2. 直接可视化

直接可视化的核心思想：从注意力矩阵中取出感兴趣的向量，reshape 成 2D 网格，resize 到图像分辨率，叠加为热力图。

### 2.1 单层单头注意力图

最基本的方式 —— 看某一层、某个 head、某个 query token（通常是 CLS token）对所有 patch 的注意力分布。

In [ ]:
# 提取第 6 层、第 3 个 head、CLS token 的注意力图
hmap_single = single_head_attention_map(
    attn=self_attn,
    layer=5,          # 第 6 层 (0-indexed)
    head=2,           # 第 3 个 head (0-indexed)
    query_token=0,    # CLS token
    grid_h=GRID_H,
    grid_w=GRID_W,
    has_cls=True,     # 第一个 token 是 CLS，要剥离
)

print(f"heatmap shape: {hmap_single.shape}")
print(f"value range: [{hmap_single.min():.4f}, {hmap_single.max():.4f}]")

# 叠加到原图
overlay_single = overlay_heatmap(img, hmap_single, colormap="jet", alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[1].imshow(hmap_single, cmap="jet", interpolation="nearest")
axes[1].set_title(f"Raw Heatmap (L5 H2)")
axes[2].imshow(np.asarray(overlay_single))
axes[2].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.suptitle("2.1 Single Head Attention Map", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.2 多头平均图

单个 head 可能噪声较大。通过对所有 head 取平均（或 max），可以看到整体趋势。

In [ ]:
# 对比三种 head 聚合方式
reductions = ["mean", "max", "min"]
fig, axes = plt.subplots(1, len(reductions) + 1, figsize=(16, 4))

axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

for i, red in enumerate(reductions):
    hmap = multi_head_average_map(
        attn=self_attn,
        layer=5,
        query_token=0,
        grid_h=GRID_H,
        grid_w=GRID_W,
        reduction=red,
    )
    overlay = overlay_heatmap(img, hmap, colormap="jet", alpha=0.5)
    axes[i + 1].imshow(np.asarray(overlay))
    axes[i + 1].set_title(f"Multi-head ({red})")
    axes[i + 1].axis("off")

plt.suptitle("2.2 Multi-Head Average Map — 不同聚合方式对比", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 多层聚合图

不同层关注的信息不同：底层关注纹理/边缘，高层关注语义。通过多层聚合可以看到综合趋势。

In [ ]:
# 聚合最后 4 层 vs 前 4 层 vs 全部层
layer_configs = {
    "First 4 layers\n(low-level)": list(range(0, 4)),
    "Last 4 layers\n(high-level)": list(range(8, 12)),
    "All 12 layers": None,
}

fig, axes = plt.subplots(1, len(layer_configs) + 1, figsize=(16, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

for i, (label, layers) in enumerate(layer_configs.items()):
    hmap = multi_layer_aggregated_map(
        attn=self_attn,
        layers=layers,
        query_token=0,
        grid_h=GRID_H,
        grid_w=GRID_W,
        head_reduction="mean",
        layer_reduction="mean",
    )
    overlay = overlay_heatmap(img, hmap, colormap="turbo", alpha=0.5)
    axes[i + 1].imshow(np.asarray(overlay))
    axes[i + 1].set_title(label)
    axes[i + 1].axis("off")

plt.suptitle("2.3 Multi-Layer Aggregated Map", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.4 文本 token 条件注意力图

在 VLM 中，cross-attention 连接文本 token 和视觉 patch。
我们可以看特定文本 token（如 "小红点"、"左上角"）对应的注意力分布，
从而理解模型是否在"看"正确的区域。

> **输入格式**: `cross_attn` 的 shape 为 `(num_heads, num_text_tokens, num_visual_tokens)`

In [ ]:
# 假设文本 tokens: ["<BOS>", "find", "the", "small", "red", "dot", "in", "the", "image", "<EOS>"]
# 我们想看 "red" (index=4) 和 "dot" (index=5) 对视觉 patch 的注意力

text_labels = ["<BOS>", "find", "the", "small", "red", "dot", "in", "the", "image", "<EOS>"]

# 使用第一层 cross-attention
cross_attn_layer0 = cross_attn_layers[0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

# 单个 token: "red" (index 4)
hmap_red = text_conditioned_attention_map(
    cross_attn=cross_attn_layer0,
    text_token_indices=4,
    grid_h=GRID_H,
    grid_w=GRID_W,
    has_cls=True,
)
axes[1].imshow(np.asarray(overlay_heatmap(img, hmap_red, alpha=0.5)))
axes[1].set_title('token: "red"')
axes[1].axis("off")

# 单个 token: "dot" (index 5)
hmap_dot = text_conditioned_attention_map(
    cross_attn=cross_attn_layer0,
    text_token_indices=5,
    grid_h=GRID_H,
    grid_w=GRID_W,
    has_cls=True,
)
axes[2].imshow(np.asarray(overlay_heatmap(img, hmap_dot, alpha=0.5)))
axes[2].set_title('token: "dot"')
axes[2].axis("off")

# 多个 token 组合: "red" + "dot"
hmap_combined = text_conditioned_attention_map(
    cross_attn=cross_attn_layer0,
    text_token_indices=[4, 5],
    grid_h=GRID_H,
    grid_w=GRID_W,
    has_cls=True,
    token_reduction="mean",  # 也可以用 "max"
)
axes[3].imshow(np.asarray(overlay_heatmap(img, hmap_combined, alpha=0.5)))
axes[3].set_title('tokens: "red"+"dot"')
axes[3].axis("off")

plt.suptitle("2.4 Text-Conditioned Cross-Attention Map", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## 3. Attention Rollout

直接看单层注意力有时会不稳定，因为信息会跨层传播。

**Attention Rollout** (Abnar & Zuidema, 2020) 通过递推聚合多层注意力来解决这个问题：

$$\hat{A}_l = 0.5 \cdot A_l + 0.5 \cdot I \quad \text{(模拟残差连接)}$$

$$R = \hat{A}_L \cdot \hat{A}_{L-1} \cdots \hat{A}_1$$

最终取 CLS token 那一行，reshape 成 2D 即为全局注意力图。

### 3.1 视觉编码器 Self-Attention Rollout

In [ ]:
# ── 不同 discard_ratio 的效果 ──────────────────────────────────
# discard_ratio 将每层最低 x% 的注意力置零，减少噪声

discard_ratios = [0.0, 0.5, 0.9]

fig, axes = plt.subplots(1, len(discard_ratios) + 1, figsize=(16, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

for i, dr in enumerate(discard_ratios):
    hmap = attention_rollout(
        attn=self_attn,
        grid_h=GRID_H,
        grid_w=GRID_W,
        has_cls=True,
        head_reduction="mean",
        discard_ratio=dr,
        query_token=0,  # CLS
    )
    overlay = overlay_heatmap(img, hmap, colormap="viridis", alpha=0.5)
    axes[i + 1].imshow(np.asarray(overlay))
    axes[i + 1].set_title(f"discard_ratio={dr}")
    axes[i + 1].axis("off")

plt.suptitle("3.1 Self-Attention Rollout — discard_ratio 对比", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 只用后几层做 rollout ───────────────────────────────────────
# 有时只关注高层语义层的注意力聚合更有意义

layer_subsets = {
    "All layers (0-11)": None,
    "Last 4 (8-11)": list(range(8, 12)),
    "Last 2 (10-11)": [10, 11],
}

fig, axes = plt.subplots(1, len(layer_subsets) + 1, figsize=(16, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

for i, (label, layers) in enumerate(layer_subsets.items()):
    hmap = attention_rollout(
        attn=self_attn,
        grid_h=GRID_H,
        grid_w=GRID_W,
        discard_ratio=0.9,
        layers=layers,
    )
    overlay = overlay_heatmap(img, hmap, colormap="hot", alpha=0.5)
    axes[i + 1].imshow(np.asarray(overlay))
    axes[i + 1].set_title(label)
    axes[i + 1].axis("off")

plt.suptitle("3.1 Self-Attention Rollout — 不同层子集", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.2 Cross-Attention Rollout

在多模态模型中，Cross-Attention Rollout 综合了：
1. 视觉编码器的 self-attention rollout（R_v）
2. 多层 cross-attention 的聚合（C）
3. （可选）文本解码器的 self-attention rollout（R_t）

最终结果 = R_tᵀ · C · R_v，表示每个文本 token 最终关注了哪些视觉区域。

In [ ]:
# ── 基本 cross-attention rollout ──────────────────────────────

# 不使用文本 rollout
hmap_cross_basic = cross_attention_rollout(
    self_attn_visual=self_attn,
    cross_attn_layers=cross_attn_layers,
    grid_h=GRID_H,
    grid_w=GRID_W,
    text_token_indices=None,  # 所有文本 token 平均
    has_cls_visual=True,
)

# 使用文本 rollout
# 先造一个文本侧的 self-attention 数据
text_self_attn = make_softmax_attn(
    (4, NUM_HEADS, NUM_TEXT_TOKENS, NUM_TEXT_TOKENS), rng
)

hmap_cross_with_text = cross_attention_rollout(
    self_attn_visual=self_attn,
    cross_attn_layers=cross_attn_layers,
    self_attn_text=text_self_attn,
    grid_h=GRID_H,
    grid_w=GRID_W,
    text_token_indices=[4, 5],  # "red" + "dot"
    has_cls_visual=True,
)

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.asarray(img))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(np.asarray(overlay_heatmap(img, hmap_cross_basic, colormap="turbo", alpha=0.5)))
axes[1].set_title("Cross-Attn Rollout\n(all text tokens)")
axes[1].axis("off")

axes[2].imshow(np.asarray(overlay_heatmap(img, hmap_cross_with_text, colormap="turbo", alpha=0.5)))
axes[2].set_title('Cross-Attn Rollout\n("red"+"dot" with text rollout)')
axes[2].axis("off")

plt.suptitle("3.2 Cross-Attention Rollout", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## 4. 保存与导出

使用 `save_visualization()` 一行代码保存叠加结果，或用 `side_by_side()` 生成对比图。

In [ ]:
# ── 保存单张 overlay 到文件 ──────────────────────────────────
saved_path = save_visualization(
    img, hmap_single,
    path="outputs/single_head_overlay.png",
    colormap="jet",
    alpha=0.5,
)
print(f"Saved to: {saved_path}")

# ── side_by_side 对比图 ──────────────────────────────────────
fig = side_by_side(
    img,
    heatmaps={
        "Single Head\n(L5 H2)": hmap_single,
        "Multi-Head\n(mean)": multi_head_average_map(
            self_attn, 5, 0, GRID_H, GRID_W
        ),
        "Self-Attn\nRollout": attention_rollout(
            self_attn, grid_h=GRID_H, grid_w=GRID_W, discard_ratio=0.9
        ),
    },
    colormap="turbo",
    alpha=0.5,
)
fig.savefig("outputs/comparison.png", dpi=150, bbox_inches="tight")
print("Saved comparison figure.")
plt.show()

---

## 5. 在真实模型上使用 (示例代码框架)

下面给出从真实 VLM 中提取注意力并调用本工具的代码框架。

**注意**：这些代码需要安装对应的模型库才能运行，此处仅作参考。

In [ ]:
# ══════════════════════════════════════════════════════════════
# 示例 1: 从 HuggingFace ViT 提取 self-attention
# ══════════════════════════════════════════════════════════════

example_vit_code = """
from transformers import ViTModel, ViTFeatureExtractor
from PIL import Image
import torch
import numpy as np

# 1. 加载模型 (output_attentions=True 非常关键!)
model = ViTModel.from_pretrained("google/vit-base-patch16-224",
                                  output_attentions=True)
extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224")

# 2. 前向推理
image = Image.open("your_image.jpg")
inputs = extractor(images=image, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# 3. 提取注意力
# outputs.attentions 是一个 tuple，每层一个 tensor
# 每个 tensor shape: (batch, num_heads, seq_len, seq_len)
attn_list = [a.squeeze(0).cpu().numpy() for a in outputs.attentions]
attn = np.stack(attn_list, axis=0)  # (num_layers, num_heads, seq_len, seq_len)
print(f"Attention shape: {attn.shape}")  # e.g. (12, 12, 197, 197)

# 4. 可视化!
from finesightbench.visualization import attention_rollout, overlay_heatmap
hmap = attention_rollout(attn, grid_h=14, grid_w=14, discard_ratio=0.9)
result = overlay_heatmap(image, hmap, colormap="turbo", alpha=0.5)
result.save("rollout_result.png")
"""

print(example_vit_code)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 示例 2: 从 LLaVA / InternVL 等 VLM 提取 cross-attention
# ══════════════════════════════════════════════════════════════

example_vlm_code = """
# ---- 模型推理时启用 output_attentions ----
# 具体方法取决于模型实现，常见做法:
#   model.config.output_attentions = True
#   或在 generate() 中传入 output_attentions=True

# ---- 提取 cross-attention ----
# 不同模型的 cross-attention 位置不同，以下是常见模式:

# 方式 A: 模型直接输出 cross_attentions
cross_attn_list = outputs.cross_attentions  # tuple of tensors
# 每个 shape: (batch, num_heads, text_len, visual_len)
cross_layers = [
    ca.squeeze(0).cpu().numpy()  # (H, T, V)
    for ca in cross_attn_list
]

# 方式 B: 联合自注意力矩阵中提取 cross-attention 子块
# 如果 VLM 把 visual + text tokens 拼接后做 joint attention:
#   full_attn shape: (L, H, V+T, V+T)
#   cross 部分 = full_attn[:, :, V:, :V]  # text→visual
#   text-self 部分 = full_attn[:, :, V:, V:]  # text→text

# ---- 调用可视化 ----
from finesightbench.visualization import (
    cross_attention_rollout,
    text_conditioned_attention_map,
    overlay_heatmap,
)

# 全局 rollout
hmap = cross_attention_rollout(
    self_attn_visual=visual_self_attn,   # (L_v, H, V, V)
    cross_attn_layers=cross_layers,       # list of (H, T, V)
    self_attn_text=text_self_attn,        # (L_t, H, T, T), optional
    grid_h=14, grid_w=14,
    text_token_indices=[idx_of_target_word],
)
result = overlay_heatmap(original_image, hmap)
result.save("cross_attn_rollout.png")

# 直接可视化特定 token
hmap_direct = text_conditioned_attention_map(
    cross_attn=cross_layers[0],  # 某一层的 (H, T, V)
    text_token_indices=[idx_of_target_word],
    grid_h=14, grid_w=14,
    has_cls=True,
)
result2 = overlay_heatmap(original_image, hmap_direct, colormap="viridis")
result2.save("text_conditioned.png")
"""

print(example_vlm_code)

---

## API 速查表

| 函数 | 用途 | 关键参数 |
|------|------|----------|
| `single_head_attention_map()` | 单层单头注意力热力图 | `layer`, `head`, `query_token` |
| `multi_head_average_map()` | 多头聚合（减少噪声） | `reduction` (mean/max/min) |
| `multi_layer_aggregated_map()` | 多层聚合（观察不同深度） | `layers`, `head_reduction`, `layer_reduction` |
| `text_conditioned_attention_map()` | 文本 token → 视觉区域的 cross-attention | `text_token_indices`, `token_reduction` |
| `attention_rollout()` | ViT self-attention rollout | `discard_ratio`, `layers` |
| `cross_attention_rollout()` | 多模态 cross-attention rollout | `self_attn_visual`, `cross_attn_layers`, `self_attn_text` |
| `overlay_heatmap()` | 将 heatmap 叠加到原图 | `colormap`, `alpha` |
| `side_by_side()` | 多张热力图并排对比 | `heatmaps` (dict) |
| `save_visualization()` | 一行保存叠加结果 | `path` |